# 19 — Skill Ecosystem Structure (Embedding-Based Similarity Network)

This notebook constructs an explicit **skill–skill ecosystem** using graph-derived embeddings learned in Chapter 2.  
Building on the job–skill bipartite graph and Node2Vec embeddings, the goal is to move beyond tabular skill indicators and represent how skills relate to one another in a continuous, relational space.

Skill embeddings encode the *context* in which skills appear across jobs rather than their raw frequency or binary co-occurrence. By measuring proximity in this embedding space, we obtain a similarity structure that reflects functional relatedness, proficiency progression, and shared labour-market usage patterns.

---

## Methodological Overview

The analysis proceeds in four stages:

1. **Embedding preparation**  
   Skill embeddings learned via Node2Vec are loaded and L2-normalised to ensure cosine similarity is well-defined and numerically stable.

2. **Skill–skill similarity matrix**  
   Pairwise cosine similarities are computed between all skills, producing a dense similarity matrix that quantifies proximity in embedding space.

3. **Network sparsification (top-k rule)**  
   To convert the dense matrix into a usable graph, only the top-5 most similar neighbours for each skill are retained. Self-similarities are explicitly excluded. This step preserves the strongest relationships while avoiding arbitrary global thresholds.

4. **Undirected edge list construction**  
   Directed neighbour relationships are canonicalised and deduplicated into an undirected skill–skill edge list. Each edge represents a strong contextual association between two skills, weighted by cosine similarity.

---

## Outputs

The primary output of this notebook is a **skill ecosystem graph artefact**:

- An undirected edge list (`skill_1`, `skill_2`, `similarity`)
- ~100 high-confidence edges linking the 27 curated skill groups
- Relationships reflecting functional similarity and skill-level progression rather than simple co-occurrence

This artefact serves as a structural representation of the labour-market skill landscape and is designed to be reused in later chapters for:
- identifying skill bundles and gateway skills  
- analysing specialisation and generalisation patterns  
- supporting career transition and upskilling analyses  

---

## Scope and Intent

This notebook is intentionally **descriptive and structural**.  
It does not perform skill clustering, community detection, or recommendation logic. Its purpose is to expose the latent relational geometry among skills learned from the job–skill graph, providing a transparent and reusable foundation for downstream analysis.

Chapter 2.4 completes the transition from tabular skill representations to a relational ecosystem view, closing the core structural work of Chapter 2.


## Set up

### Libraries and other imports

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path
#===
from sklearn.preprocessing import normalize

### Path

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

### Data

In [3]:
from src.job_intel.features.embedding_loader import load_node2vec_embeddings

In [4]:
_, skill_emb = load_node2vec_embeddings(tag='v01')

In [5]:
skill_emb_norm = normalize(skill_emb, norm='l2')

In [6]:
# Checks
print(f'NAs in the skill embeddings = {skill_emb.isna().sum().sum()}.')
print(f'NAs in the skill normalised embeddings = {np.isnan(skill_emb_norm).sum()}.')
print(f'Shape maintained for skill embeddigs? {skill_emb.shape == skill_emb_norm.shape}.')
print(f'Unit norm check: {np.sqrt((skill_emb_norm[:10]**2).sum(axis=1))}')

NAs in the skill embeddings = 0.
NAs in the skill normalised embeddings = 0.
Shape maintained for skill embeddigs? True.
Unit norm check: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Compute the Skill Similarity Matrix

In [7]:
skill_t = skill_emb_norm.T
mat_dot = np.dot(skill_emb_norm, skill_t)
similarity_matrix = pd.DataFrame(mat_dot)
similarity_matrix.index = skill_emb.index
similarity_matrix.columns = skill_emb.index

In [8]:
similarity_matrix

skill,core_programming__basic_prob,core_programming__intermediate_prob,core_programming__advanced_prob,data_engineering_pipelines__basic_prob,data_engineering_pipelines__intermediate_prob,data_engineering_pipelines__advanced_prob,ml_ai__basic_prob,ml_ai__intermediate_prob,ml_ai__advanced_prob,analytics_stats__basic_prob,...,cloud__advanced_prob,db_storage__basic_prob,db_storage__intermediate_prob,db_storage__advanced_prob,productivity_workflow__basic_prob,productivity_workflow__intermediate_prob,productivity_workflow__advanced_prob,soft_skills__core_prob,soft_skills__leadership_prob,domain_specific__none_prob
skill,,,,,,,,,,,,,,,,,,,,,
core_programming__basic_prob,1.000000,0.241668,0.084359,0.160249,0.263785,0.230491,0.307436,0.312532,0.252196,0.385093,...,0.202814,0.397319,0.205114,0.173261,0.216144,0.186000,0.112881,0.327922,0.231593,0.206148
core_programming__intermediate_prob,0.241668,1.000000,0.114733,0.123027,0.277812,0.233012,0.157802,0.134050,0.140979,0.160096,...,0.171369,0.195850,0.205745,0.157109,0.120996,0.153808,0.117247,0.157109,0.149336,0.116517
core_programming__advanced_prob,0.084359,0.114733,1.000000,0.092200,0.110210,0.096637,0.140334,0.083132,0.129131,0.135583,...,0.119706,0.109676,0.112607,0.111447,0.132912,0.114502,0.101473,0.108015,0.162477,0.095331
data_engineering_pipelines__basic_prob,0.160249,0.123027,0.092200,1.000000,0.151192,0.131607,0.048792,0.051559,0.070956,0.162897,...,0.118295,0.172860,0.196716,0.089625,0.080557,0.152615,0.102260,0.165986,0.152023,0.133071
data_engineering_pipelines__intermediate_prob,0.263785,0.277812,0.110210,0.151192,1.000000,0.280946,0.213306,0.171691,0.159332,0.249194,...,0.138277,0.330143,0.336831,0.182456,0.160691,0.251077,0.116497,0.185070,0.294175,0.133615
data_engineering_pipelines__advanced_prob,0.230491,0.233012,0.096637,0.131607,0.280946,1.000000,0.172690,0.185605,0.202348,0.207561,...,0.199753,0.196996,0.248312,0.169358,0.232703,0.169190,0.072049,0.150253,0.216755,0.164692
ml_ai__basic_prob,0.307436,0.157802,0.140334,0.048792,0.213306,0.172690,1.000000,0.392079,0.356445,0.265983,...,0.193717,0.188812,0.121004,0.120826,0.158249,0.116795,0.125684,0.229210,0.159719,0.171220
ml_ai__intermediate_prob,0.312532,0.134050,0.083132,0.051559,0.171691,0.185605,0.392079,1.000000,0.404061,0.261415,...,0.208615,0.182056,0.155254,0.121404,0.120999,0.127850,0.107343,0.241287,0.165639,0.200939
ml_ai__advanced_prob,0.252196,0.140979,0.129131,0.070956,0.159332,0.202348,0.356445,0.404061,1.000000,0.190211,...,0.252264,0.117804,0.130242,0.118584,0.171156,0.140469,0.137641,0.186798,0.130138,0.147393


### Checks

In [9]:
print(f"Similarity matrix shape = {mat_dot.shape}")

# Symmetry check (tolerance-based, correct for floats)
is_symmetric = np.allclose(mat_dot, mat_dot.T, atol=1e-12, rtol=1e-12)
print(f"Symmetry check (allclose) = {is_symmetric}")

diag = np.diagonal(mat_dot)
print(f"Average diagonal value {diag.mean()}")
print(f"Min diagonal value {diag.min()}")
print(f"Max diagonal value {diag.max()}")


Similarity matrix shape = (27, 27)
Symmetry check (allclose) = True
Average diagonal value 1.0
Min diagonal value 0.9999999999999993
Max diagonal value 1.0000000000000007


### Sparsify the similarity matrix into a network (edge list)

In [10]:
# set diagonal to -inf
np.fill_diagonal(similarity_matrix.values, -np.inf)
similarity_matrix

skill,core_programming__basic_prob,core_programming__intermediate_prob,core_programming__advanced_prob,data_engineering_pipelines__basic_prob,data_engineering_pipelines__intermediate_prob,data_engineering_pipelines__advanced_prob,ml_ai__basic_prob,ml_ai__intermediate_prob,ml_ai__advanced_prob,analytics_stats__basic_prob,...,cloud__advanced_prob,db_storage__basic_prob,db_storage__intermediate_prob,db_storage__advanced_prob,productivity_workflow__basic_prob,productivity_workflow__intermediate_prob,productivity_workflow__advanced_prob,soft_skills__core_prob,soft_skills__leadership_prob,domain_specific__none_prob
skill,,,,,,,,,,,,,,,,,,,,,
core_programming__basic_prob,-inf,0.241668,0.084359,0.160249,0.263785,0.230491,0.307436,0.312532,0.252196,0.385093,...,0.202814,0.397319,0.205114,0.173261,0.216144,0.186000,0.112881,0.327922,0.231593,0.206148
core_programming__intermediate_prob,0.241668,-inf,0.114733,0.123027,0.277812,0.233012,0.157802,0.134050,0.140979,0.160096,...,0.171369,0.195850,0.205745,0.157109,0.120996,0.153808,0.117247,0.157109,0.149336,0.116517
core_programming__advanced_prob,0.084359,0.114733,-inf,0.092200,0.110210,0.096637,0.140334,0.083132,0.129131,0.135583,...,0.119706,0.109676,0.112607,0.111447,0.132912,0.114502,0.101473,0.108015,0.162477,0.095331
data_engineering_pipelines__basic_prob,0.160249,0.123027,0.092200,-inf,0.151192,0.131607,0.048792,0.051559,0.070956,0.162897,...,0.118295,0.172860,0.196716,0.089625,0.080557,0.152615,0.102260,0.165986,0.152023,0.133071
data_engineering_pipelines__intermediate_prob,0.263785,0.277812,0.110210,0.151192,-inf,0.280946,0.213306,0.171691,0.159332,0.249194,...,0.138277,0.330143,0.336831,0.182456,0.160691,0.251077,0.116497,0.185070,0.294175,0.133615
data_engineering_pipelines__advanced_prob,0.230491,0.233012,0.096637,0.131607,0.280946,-inf,0.172690,0.185605,0.202348,0.207561,...,0.199753,0.196996,0.248312,0.169358,0.232703,0.169190,0.072049,0.150253,0.216755,0.164692
ml_ai__basic_prob,0.307436,0.157802,0.140334,0.048792,0.213306,0.172690,-inf,0.392079,0.356445,0.265983,...,0.193717,0.188812,0.121004,0.120826,0.158249,0.116795,0.125684,0.229210,0.159719,0.171220
ml_ai__intermediate_prob,0.312532,0.134050,0.083132,0.051559,0.171691,0.185605,0.392079,-inf,0.404061,0.261415,...,0.208615,0.182056,0.155254,0.121404,0.120999,0.127850,0.107343,0.241287,0.165639,0.200939
ml_ai__advanced_prob,0.252196,0.140979,0.129131,0.070956,0.159332,0.202348,0.356445,0.404061,-inf,0.190211,...,0.252264,0.117804,0.130242,0.118584,0.171156,0.140469,0.137641,0.186798,0.130138,0.147393


In [11]:
res = []
skills = similarity_matrix.index
n = len(skills)

for i in range(n):
    
    top_5 = similarity_matrix.iloc[i,:].nlargest(5)

    for j in range(5):

        out = { 'skill_source': skills[i],
                'skill_target':top_5.index[j],
                'similarity':top_5.iloc[j]}
        res.append(out)

top_5_skill_neighbours = pd.DataFrame(res)
    

In [12]:
print('Output check')
print(f'Shape: {top_5_skill_neighbours.shape}')
print(f'Confirm no self edges: {sum(top_5_skill_neighbours['skill_source'] == top_5_skill_neighbours['skill_target'])}')
print(f'Inspect top 10: {top_5_skill_neighbours.head(10)}')


Output check
Shape: (135, 3)
Confirm no self edges: 0
Inspect top 10:                           skill_source  \
0         core_programming__basic_prob   
1         core_programming__basic_prob   
2         core_programming__basic_prob   
3         core_programming__basic_prob   
4         core_programming__basic_prob   
5  core_programming__intermediate_prob   
6  core_programming__intermediate_prob   
7  core_programming__intermediate_prob   
8  core_programming__intermediate_prob   
9  core_programming__intermediate_prob   

                                    skill_target  similarity  
0                         db_storage__basic_prob    0.397319  
1                    analytics_stats__basic_prob    0.385093  
2                         soft_skills__core_prob    0.327922  
3                       ml_ai__intermediate_prob    0.312532  
4                              ml_ai__basic_prob    0.307436  
5                              cloud__basic_prob    0.281071  
6  data_engineering_pipeli

### Deduplicate

In [13]:
top_5_skill_neighbours['skill_1']=top_5_skill_neighbours[['skill_source', 'skill_target']].min(axis=1)
top_5_skill_neighbours['skill_2']=top_5_skill_neighbours[['skill_source', 'skill_target']].max(axis=1)
top_5_skill_neighbours = top_5_skill_neighbours.groupby(by =['skill_1', 'skill_2']).agg(similarity = ('similarity', 'max')).reset_index()

In [14]:
top_5_skill_neighbours

,skill_1,skill_2,similarity
0,analytics_stats__advanced_prob,bi_viz__intermediate_prob,0.177396
1,analytics_stats__advanced_prob,core_programming__basic_prob,0.208793
2,analytics_stats__advanced_prob,db_storage__basic_prob,0.190524
3,analytics_stats__advanced_prob,ml_ai__basic_prob,0.199507
4,analytics_stats__advanced_prob,ml_ai__intermediate_prob,0.175915
...,...,...,...
96,productivity_workflow__advanced_prob,soft_skills__core_prob,0.216783
97,productivity_workflow__basic_prob,productivity_workflow__intermediate_prob,0.224466
98,productivity_workflow__basic_prob,soft_skills__leadership_prob,0.219887
99,productivity_workflow__intermediate_prob,soft_skills__leadership_prob,0.241931


In [15]:
top_5_skill_neighbours.sort_values('similarity', ascending=False).head(10)

,skill_1,skill_2,similarity
100,soft_skills__core_prob,soft_skills__leadership_prob,0.463077
16,analytics_stats__basic_prob,soft_skills__core_prob,0.427088
34,bi_viz__intermediate_prob,db_storage__basic_prob,0.407954
90,ml_ai__advanced_prob,ml_ai__intermediate_prob,0.404061
63,core_programming__basic_prob,db_storage__basic_prob,0.397319
92,ml_ai__basic_prob,ml_ai__intermediate_prob,0.392079
9,analytics_stats__basic_prob,core_programming__basic_prob,0.385093
85,db_storage__basic_prob,db_storage__intermediate_prob,0.375845
17,analytics_stats__basic_prob,soft_skills__leadership_prob,0.371323
27,bi_viz__basic_prob,bi_viz__intermediate_prob,0.356539


In [16]:
top_5_skill_neighbours.shape

(101, 3)

### Save output

In [17]:
from src.job_intel.config import PROCESSED_DATA_DIR
top_5_skill_neighbours.to_csv(PROCESSED_DATA_DIR / "skill_similarity_edges_k5_embeddings.csv", index=False)

### Interpretation
The skill–skill network derived from Node2Vec embeddings reveals a coherent ecosystem structure in which skills cluster primarily by functional proximity and proficiency progression. The strongest connections link adjacent skill levels within the same domain (e.g., basic → intermediate ML, BI visualisation, database storage) and closely related transversal competencies (e.g., core soft skills and leadership). This indicates that the embedding space captures higher-order contextual similarity rather than simple co-occurrence, supporting its use as a relational representation of the labour-market skill landscape.

# == End of Notebook == 